# Brainstorming and Focus Group Quantitative Experimentation 2.1: :**Difficult people** under **divergence intervention** only

Can we use TinyTroupe to brainstorm product ideas?

In [1]:
import sys

from pprint import pprint

from tinytroupe.agent import TinyPerson
from tinytroupe.environment import TinyWorld
from tinytroupe.experimentation import InPlaceExperimentRunner
from tinytroupe.steering import Intervention
from tinytroupe.examples import *
from tinytroupe.validation import propositions
from tinytroupe.extraction import ResultsExtractor
from tinytroupe.utils.parallel import parallel_map_dict, parallel_map_cross
from tinytroupe.validation import hard_persona_adherence, persona_adherence, self_consistency, fluency, task_completion, divergence

# specific utilities
from common_utils import *


!!!!
DISCLAIMER: TinyTroupe relies on Artificial Intelligence (AI) models to generate content. 
The AI models are not perfect and may produce inappropriate or inaccurate results. 
For any serious or consequential use, please review the generated content before using it.
!!!!

Looking for default config on: C:\Users\pdasilva\repos\TinyTroupe and personal repos\TinyTroupe\tinytroupe\utils\..\config.ini
Found custom config on: c:\Users\pdasilva\repos\TinyTroupe and personal repos\TinyTroupe\publications\paper_artifacts_april-2026\config.ini
TinyTroupe version: 0.8.0
Current date and time (local): 2026-05-03 01:08:41
Current date and time (UTC):   2026-05-03 04:08:41

Current TinyTroupe configuration 
[OpenAI]
api_type = azure
azure_api_version = 2024-12-01-preview
model = gpt-5-mini
reasoning_model = o3-mini
vision_detail = auto
embedding_model = text-embedding-3-small
azure_embedding_model_api_version = 2023-05-15
max_completion_tokens = 128000
timeout = 300
max_attempts = 5
waiting_tim

## Parameters

In [2]:
full_mode = True  # set to True to run the full mode with all agents and tasks

# avoid displaying the communication, to make the output cleaner for eval
TinyPerson.communication_display = False

In [3]:
if full_mode:
    repetitions_per_task = 2
    simulation_steps = 5
    qty_agents = 12
    qty_proposals = 4

else:
    repetitions_per_task = 2
    simulation_steps = 5
    qty_agents = 4
    qty_proposals = 1


## Experiment setup

In [4]:
experiment_runner = InPlaceExperimentRunner("./brainstorming_and_focus_group_quantitative_experimentation_2.2.json")

experiment_runner.add_experiment("Control")
experiment_runner.add_experiment("Treatment")

2026-05-03 01:10:04,809 - MainThread(41284) - tinytroupe - WARNING - Configuration file './brainstorming_and_focus_group_quantitative_experimentation_2.2.json' exists and was loaded successfully. If you are trying to fully rerun the experiments, delete it first.
2026-05-03 01:10:04,840 - MainThread(41284) - tinytroupe - INFO - Experiment 'Control' already exists, nothihg to add.
2026-05-03 01:10:04,840 - MainThread(41284) - tinytroupe - INFO - Experiment 'Control' already exists, nothihg to add.
2026-05-03 01:10:04,843 - MainThread(41284) - tinytroupe - INFO - Experiment 'Treatment' already exists, nothihg to add.


In [5]:
experiment_runner.activate_next_experiment()

#experiment_runner.fix_active_experiment("Control")
#experiment_runner.fix_active_experiment("Treatment")

In [6]:
print(f"Running experiment {experiment_runner.get_active_experiment()}")

Running experiment Treatment


## Agents and populations

In [7]:

people = []
if not experiment_runner.has_finished_all_experiments():
    # load agents
    people = TinyPerson.load_specifications_from_folder("./population/difficult_people_2")

    # filter to make it go faster?
    if qty_agents is not None:
        people = people[:qty_agents]

    # customize and print minibios 
    for person in people:

        person.import_fragment("./fragments/difficult_person.agent.fragment.json")

        # disable quality checks for both Control and Treatment
        person.action_generator.enable_quality_checks = False

        print(person.minibio(extended=False))


Alan Merrick is a 48 year old Administrative Officer (Benefits and Records), British, currently living in Manchester, United Kingdom.
Anthony Russo is a 42 year old Journeyman Electrician / Senior Field Technician, American, currently living in Cleveland, Ohio, USA.
Anya Calder-Mori is a 45 year old Freelance Graphic Designer, Conceptual Artist and Cultural Critic, British, currently living in Camberwell, London, UK.
Barbara Jean Pratt is a 68 year old Retiree (former assembly line worker / part-time volunteer at church thrift shop), American, currently living in Small town near Toledo, Ohio, USA.
Colin Arthur Matthews is a 42 year old Operations Manager (Mid-level), British, currently living in Manchester, UK.
Colin Murray is a 52 year old Benefits and Housing Support Officer, British, currently living in Salford, Greater Manchester, UK.
Connor Walsh is a 28 year old Senior Customer Service Associate / Shift Lead (Retail Grocery Chain), American, currently living in Cleveland, Ohio, U

In [8]:
len(people)

12

In [9]:
# divide people in several groups of 5
people_groups = []
for i in range(0, len(people), 4):
    people_groups.append(people[i:i+4]
    )

len(people_groups)

3

In [10]:
# In this experiment, we'll not use action correction. We'll instead experiment only with divergence intervention.
for person in people:
    person.action_generator.enable_reasoning_step = False
    person.action_generator.enable_quality_checks = False

## Proposals

In [11]:
proposals = [
    {"theme": "Daily Life and Convenience",
     "objective": "Create ideas for products or services that simplify, enhance, or bring joy to everyday tasks, routines, and interactions."},

    {"theme": "Personal Growth and Wellbeing",
     "objective": "Generate concepts for products or experiences that support personal development, health, mental wellness, emotional care, or community connection."},

    {"theme": "Discovery and Exploration",
     "objective": "Explore ideas for products, services, or platforms that encourage curiosity, learning, adventure, and exploration of both the external world and inner self."},

    {"theme": "Productivity and Resourcefulness",
     "objective": "Invent new tools, processes, or organizational systems that empower people or groups to achieve more, optimize resources, or collaborate effectively."},

    {"theme": "Creativity and Expression",
     "objective": "Design ideas for new products, platforms, or services that inspire creativity, foster expression, enhance artistic skills, or enable new forms of storytelling and communication."}
]

if not full_mode:
    proposals = proposals[:qty_proposals]

In [12]:
# divide the proposals in exactly two groups (half/half)
proposals_groups = []
proposals_groups.append(proposals[:len(proposals)//2])
proposals_groups.append(proposals[len(proposals)//2:])

proposals_groups

[[{'theme': 'Daily Life and Convenience',
   'objective': 'Create ideas for products or services that simplify, enhance, or bring joy to everyday tasks, routines, and interactions.'},
  {'theme': 'Personal Growth and Wellbeing',
   'objective': 'Generate concepts for products or experiences that support personal development, health, mental wellness, emotional care, or community connection.'}],
 [{'theme': 'Discovery and Exploration',
   'objective': 'Explore ideas for products, services, or platforms that encourage curiosity, learning, adventure, and exploration of both the external world and inner self.'},
  {'theme': 'Productivity and Resourcefulness',
   'objective': 'Invent new tools, processes, or organizational systems that empower people or groups to achieve more, optimize resources, or collaborate effectively.'},
  {'theme': 'Creativity and Expression',
   'objective': 'Design ideas for new products, platforms, or services that inspire creativity, foster expression, enhance art

## Auxiliary functions

In [13]:
def brainstorming_battery(agents, proposals, interventions, agent_propositions, environment_propositions, 
                          repetitions = 5, simulation_steps=10): 
    
    agent_propositions_scores = {}
    environment_propositions_scores = {}

    experiments_count = 0
    total_expected_experiments = len(proposals) * repetitions #* len(agents)

    # loop over proposals and repetitions
    for proposal in proposals:

        objective = proposal["objective"]
        theme = proposal["theme"]

        for i in range(repetitions):
            print("\n############## STARTING A NEW RESEARCH SESSION #################")
            print(f"Overall experiment number: {experiments_count+1} / {total_expected_experiments}")
            print(f"Discussion objective: {objective}")
            print(f"Trial number: {i+1}")
            print(f"Agents: {agents}")

            # clear the episodic memory of all agents
            for person in agents:
                person.clear_episodic_memory()

            world = TinyWorld(agents=agents, interventions=interventions)
            
            # Participants introduce themselves
            world.broadcast(f"""
                Hello everyone! Let's start by introducing ourselves, and mentioning problems we face in our daily personal
                and professional lives related to the following theme: {theme}
                
                Please:
                  - present yourself and your background;
                  - present some key personal problems related to the theme;
                  - present some key problems related to the theme that you face in your work;
                  - present some key problems related to the theme that you see in your industry as a whole.
                  
                Don't discuss solutions yet, just the problems you face and see others facing.
                """)
            world.run(1)
            
            # now to the brainstorming session itself
            world.broadcast(f"""
                Folks, your mission is to brainstorm {objective}. 
                Please follow these guidelines:
                  - give a unique and informative name to each idea you propose, so that it is easy to refer to it. Say it like "Idea name: '<name of the idea>'".;
                  - explain why you think it is a good idea, and what problem it solves, and how you feel about it;
                  - your ideas should be new complete, self-contained, products or services, not features for other existing products or services;
                  - think of creative ideas that would somehow help you in both in your personal and professional lives.
                  - create as many different and unique ideas as you can during the brainstorming session. Each idea must be **completely** different from the others 
                    (either by yourself or by others), and not just a variation of an existing idea.                    
                  - you should criticize each other's ideas, in order to make sure they are as
                    good as possible, but no more than once per idea.
                  - you should also provide suggestions for improvement to each other's ideas, in order to make them as good as possible, 
                    but no more than once per idea.
                  - regardless of critique or complement, you **must** primarily propose new ideas quickly, 
                    not just build on existing ones. 
                  - propose one idea at a time, instead of proposing multiple ideas at once, to allow appropriate discussion.
                  - you should **not** propose ideas that are too similar to each other, or to the ones already proposed by others.
                  - before saying anything, THINK deeply about yourself, your beliefs, interests, needs, life, etc., to come up with ideas that are
                    truly unique and different from the ones already proposed by others.
                   
                Please start the discussion now.
                """)
            world.run(simulation_steps)

            # extract and count ideas
            rapporteur = agents[0]  # the first agent is the rapporteur
            rapporteur.listen_and_act("Can you please consolidate the ideas that the group came up with? Provide a lot of details on each idea, and complement anything missing.")
            ideas = ResultsExtractor().extract_results_from_agent(rapporteur, 
                                    extraction_objective="Consolidates the ideas that the group came up with, explaining each idea as an item of a list." \
                                                        "Add information about: what problem the idea solves; to which target audience it is meant." \
                                                        "how is it different from competing, existing, products.", 
                                    situation="A focus group to brainstorm new product ideas.",
                                    fields= ["name", "description", "problem", "target_audience", "competition_analysis"],
                                    fields_hints={"ideas": "must be the root of the resulting dictionary."},)
            pprint(ideas)
            if "ideas_qty" not in environment_propositions_scores:
                environment_propositions_scores["ideas_qty"] = []
            if ideas is not None and "ideas" in ideas and isinstance(ideas["ideas"], list):
                environment_propositions_scores["ideas_qty"].append(len(ideas["ideas"]))

            # Evaluate environment propositions in parallel
            env_results = parallel_map_dict(
                environment_propositions,
                lambda item: item[1].copy().score(
                    world, 
                    claim_variables={"task_description": f"A brainstorming or focus group session was run about: {objective}."}, 
                    return_full_response=True
                )
            )
            
            # Process environment results
            for k, result in env_results.items():
                if k not in environment_propositions_scores:
                    environment_propositions_scores[k] = []
                environment_propositions_scores[k].append(result["value"])
                print("value: ", result["value"])
                print("justification: ", result["justification"])
                print("reasoning: ", result["reasoning"])

            # Evaluate agent propositions across all agents in parallel
            agent_results = parallel_map_cross(
                [agents, agent_propositions.items()],
                lambda agent, prop_item: (
                    prop_item[0],  # proposition key
                    prop_item[1].copy().score(agent, return_full_response=True)  # result
                )
            )
            
            # Process agent results
            for k, result in agent_results:
                if k not in agent_propositions_scores:
                    agent_propositions_scores[k] = []
                if result is not None:
                    agent_propositions_scores[k].append(result["value"])
                    print("value: ", result["value"])
                    print("justification: ", result["justification"])
                    print("reasoning: ", result["reasoning"])
                    print("\n\n")
                else:
                    print(f"*****WARNING:***** Agent did not respond to proposition {k}.")
            #
            ##for k, proposition in agent_propositions.items():
            ##    for person in world.agents:
            ##        result = proposition.copy().score(person, return_full_response=True)
            ##        
            ##        if k not in agent_propositions_scores:
            ##            agent_propositions_scores[k] = []
            ##        agent_propositions_scores[k].append(result["value"])
            ##
            ##        print("value: ", result["value"])
            ##        print("justification: ", result["justification"])
            ##        print("reasoning: ", result["reasoning"])
            ##        print("\n\n")
            ##
            
            experiments_count += 1
            print("\n\n")

    return agent_propositions_scores, environment_propositions_scores



## Perform experiment

In [14]:
agent_propositions_scores={}
environment_propositions_scores={}

In [15]:
def brainstorm(people, proposals=proposals):
    global agent_propositions_scores, environment_propositions_scores
    if not experiment_runner.has_finished_all_experiments():

        interventions = []
        if experiment_runner.get_active_experiment() == "Treatment":
            interventions = \
                Intervention.create_for_each(people)\
                    .set_functional_precondition(lambda target: target.actions_count >=7)\
                    .set_textual_precondition(
                        """
                        AGENT IS NOT PROPOSING COMPLETELY NEW PRODUCT/SERVICE IDEAS ANYMORE:
                        The last **entirely** new product/service idea proposed by this agent, if any, was proposed by him/her **more** than 5 of simulation events ago.
                        That is to say, the agent has not proposed any new product/service idea in the last 5 of his/her simulation trajectory events.
                        Additional features, variations of or other refinements to product/service ideas already proposed are NOT considered new!

                        How to compute the steps gap:
                        1. Determine the current next event number (N); and the last event number in which the agent proposed a new product/service idea (M).
                            This information can be found in the simulation trajectory.
                        2. Compute the **difference** beteween the current next event number and the last event number in which the agent proposed a new product/service idea: D = N - M
                        3. The proposition is true if, and only if, the difference D is **greater than** 5.
                        """)\
                    .set_effect(lambda target: target.think("""
                                                            I need to propose additional, **completelly** new and different, product/service ideas. This was part of the requirement for this session.
                                                            I will propose an entirely **new** idea now, I **cannot** repeat or refine previous ideas! I cannot make variations
                                                            of previous ideas (e.g., "XYZ for A", "XYZ for B", "XYZ for Z" are repetitive, there should be only one "XYZ"), 
                                                            I need to think of something **entirely** new and different.
                                                            To help me avoid repeating previous ideas, I'll now explicitly THINK about all the ideas already given by myself or
                                                            others, and then, based on that, I'll think again about a new unique idea.
                                                            """))

                                                            
        tmp_agent_propositions_scores, tmp_environment_propositions_scores = \
            brainstorming_battery(
                agents=people,
                proposals=proposals,
                interventions=interventions,    
                agent_propositions={
                    "Hard Persona Adherence": hard_persona_adherence,
                    "Self-consistency": self_consistency,
                    "Fluency": fluency
                },
                environment_propositions={
                    "Task Completion": task_completion,
                    "Divergence": divergence
                },
                repetitions=repetitions_per_task,
                simulation_steps=simulation_steps
            )

        pprint("NEW AGENT PROPOSITIONS SCORES")
        pprint(tmp_agent_propositions_scores)
        print("\n\n")
        pprint("NEW ENVIRONMENT PROPOSITIONS SCORES")
        pprint(tmp_environment_propositions_scores)

        # merge the scores lists
        agent_propositions_scores = merge_dicts_of_lists(tmp_agent_propositions_scores, agent_propositions_scores)
        environment_propositions_scores = merge_dicts_of_lists(tmp_environment_propositions_scores, environment_propositions_scores)

        return agent_propositions_scores, environment_propositions_scores

In [16]:
brainstorm(people_groups[0], proposals_groups[0]) if len(people_groups) > 0  and len(proposals_groups) > 0 else None


############## STARTING A NEW RESEARCH SESSION #################
Overall experiment number: 1 / 4
Discussion objective: Create ideas for products or services that simplify, enhance, or bring joy to everyday tasks, routines, and interactions.
Trial number: 1
Agents: [TinyPerson(name='Alan Merrick'), TinyPerson(name='Anthony Russo'), TinyPerson(name='Anya Calder-Mori'), TinyPerson(name='Barbara Jean Pratt')]
2026-05-03 01:10:08,845 - MainThread(41284) - tinytroupe - INFO - [TinyWorld 1] Running world simulation step 1 of 1.


───────────────────────────────────────────── TinyWorld 1 step 1 of 1 ─────────────────────────────────────────────

2026-05-03 01:10:08,866 - MainThread(41284) - tinytroupe - INFO - [TinyWorld 1] No timedelta provided, so the datetime was not advanced.
2026-05-03 01:10:10,940 - MainThread(41284) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 01:10:11,586 - MainThread(41284) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-03 01:10:37,485 - MainThread(41284) - httpx - INFO - HTTP Request: POST https://azureai-prototyping-ai.openai.azure.com/openai/deployments/gpt-5-mini/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-05-03 01:10:38,141 - MainThread(41284) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 01:10:38,148 - MainThread(41284) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-03 01:11:08,131 - MainThread(41284) - httpx - INFO - HTTP Request: POST https://azureai-prototyping-ai.openai.azure.com/openai/deployments/gpt-

───────────────────────────────────────────── TinyWorld 1 step 1 of 5 ─────────────────────────────────────────────

2026-05-03 01:16:08,933 - MainThread(41284) - tinytroupe - INFO - [TinyWorld 1] No timedelta provided, so the datetime was not advanced.
2026-05-03 01:16:10,669 - MainThread(41284) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 01:16:10,689 - MainThread(41284) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-03 01:16:48,224 - MainThread(41284) - httpx - INFO - HTTP Request: POST https://azureai-prototyping-ai.openai.azure.com/openai/deployments/gpt-5-mini/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-05-03 01:16:49,598 - MainThread(41284) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 01:16:49,622 - MainThread(41284) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-03 01:17:12,601 - MainThread(41284) - httpx - INFO - HTTP Request: POST https://azureai-prototyping-ai.openai.azure.com/openai/deployments/gpt-

...

 ⚡  <<Intervention 3>> Triggered, effects are being applied... 
          > The proposition is true because: (1) the simulation explicitly states the next event
          > number N = 44; (2) the last event in which Anya Calder-Mori proposed an entirely new
          > product/service idea is event #34 where she posted Idea name: 'Ritual Press' (a
          > distinct, complete product/service); (3) subsequent Anya events (#35 DONE, #39 THINK,
          > #40 TALK, #41 DONE) contain no new idea proposals — only commentary, reflections, or
          > signaling waiting states; (4) computing the gap gives D = 44 - 34 = 10, which is greater
          > than 5; (5) therefore the agent has not proposed any entirely new product/service idea
          > in the last 5 of her simulation trajectory events, satisfying the proposition. I
          > explicitly note that earlier Anya proposals (e.g., #21 'ProofKit') are older than #34
          > and that later messages are either refinements/comments, not new ideas, consistent with
          > the proposition's rule that refinements do not count as new. (confidence = 1.0)
          > Functional precondition was met.

2026-05-03 02:35:52,917 - MainThread(41284) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 02:35:52,940 - MainThread(41284) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-03 02:36:24,273 - MainThread(41284) - httpx - INFO - HTTP Request: POST https://azureai-prototyping-ai.openai.azure.com/openai/deployments/gpt-5-mini/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"


 ⚡  <<Intervention 4>> Triggered, effects are being applied... 
          > The proposition is true because the most recent entirely new product/service idea
          > proposed by Barbara Jean Pratt occurred at event #35 (Idea name: "Commons Tool Chest").
          > The current next event number is N = 46 (explicitly given in the context). Using the
          > required computation D = N - M gives D = 46 - 35 = 11. The rule states the proposition
          > is true iff D > 5; here 11 > 5, so the condition holds. Concrete evidence from the
          > trajectory: - Event #22: Barbara proposed 'Neighborhood Fix‑It Caravan' (a full, self-
          > contained idea). - Event #35: Barbara proposed 'Commons Tool Chest' (a distinct, self-
          > contained idea). - Events after #35 (events #36–#45) show follow-ups, DONE markers, and
          > responses (Alan, Anthony, Anya) and Barbara providing critiques and clarifications
          > (e.g., events #40–#41 are Barbara's critique of Anya's idea), but Barbara does not
          > propose any new, entirely distinct product/service idea after #35. The context also
          > clarifies that refinements, variations, or critiques (for example her suggestions on
          > ProofKit, Ritual Press, or comments on others' ideas) do not count as new ideas per the
          > proposition's rules. Thus all required elements (N, M, D calculation, and interpretation
          > against the threshold) support the truth of the proposition. (confidence = 1.0)
          > Functional precondition was met.

2026-05-03 02:36:26,742 - ThreadPoolExecutor-29_1(15236) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 02:36:26,761 - ThreadPoolExecutor-29_0(36728) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 02:36:26,800 - ThreadPoolExecutor-29_3(19848) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 02:36:26,841 - ThreadPoolExecutor-29_2(25448) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 02:36:26,869 - ThreadPoolExecutor-29_1(15236) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-03 02:36:26,880 - ThreadPoolExecutor-29_0(36728) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-03 02:36:26,922 - ThreadPoolExecutor-29_3(19848) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-03 02:36:26,966 - ThreadPoolExecutor-29_2(25448) - tinytroup

({'Hard Persona Adherence': [2, 2, 0, 4, 1, 4, 0, 1, 0, 3, 2, 1, 0, 1, 2, 2],
  'Self-consistency': [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9],
  'Fluency': [8, 7, 8, 8, 7, 8, 7, 8, 6, 6, 6, 5, 7, 7, 1, 7]},
 {'ideas_qty': [12, 12, 15, 12],
  'Task Completion': [9, 9, 9, 9],
  'Divergence': [8, 2, 8, 4]})

In [17]:
brainstorm(people_groups[0], proposals_groups[1]) if len(people_groups) > 0  and len(proposals_groups) > 1 else None


############## STARTING A NEW RESEARCH SESSION #################
Overall experiment number: 1 / 6
Discussion objective: Explore ideas for products, services, or platforms that encourage curiosity, learning, adventure, and exploration of both the external world and inner self.
Trial number: 1
Agents: [TinyPerson(name='Alan Merrick'), TinyPerson(name='Anthony Russo'), TinyPerson(name='Anya Calder-Mori'), TinyPerson(name='Barbara Jean Pratt')]
2026-05-03 02:48:20,936 - MainThread(41284) - tinytroupe - INFO - [TinyWorld 5] Running world simulation step 1 of 1.


───────────────────────────────────────────── TinyWorld 5 step 1 of 1 ─────────────────────────────────────────────

2026-05-03 02:48:20,944 - MainThread(41284) - tinytroupe - INFO - [TinyWorld 5] No timedelta provided, so the datetime was not advanced.
2026-05-03 02:48:21,806 - MainThread(41284) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 02:48:21,814 - MainThread(41284) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-03 02:48:49,660 - MainThread(41284) - httpx - INFO - HTTP Request: POST https://azureai-prototyping-ai.openai.azure.com/openai/deployments/gpt-5-mini/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-05-03 02:48:50,400 - MainThread(41284) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 02:48:50,405 - MainThread(41284) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-03 02:49:21,390 - MainThread(41284) - httpx - INFO - HTTP Request: POST https://azureai-prototyping-ai.openai.azure.com/openai/deployments/gpt-

───────────────────────────────────────────── TinyWorld 5 step 1 of 5 ─────────────────────────────────────────────

2026-05-03 02:51:13,082 - MainThread(41284) - tinytroupe - INFO - [TinyWorld 5] No timedelta provided, so the datetime was not advanced.
2026-05-03 02:51:13,927 - MainThread(41284) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 02:51:13,937 - MainThread(41284) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-03 02:51:53,200 - MainThread(41284) - httpx - INFO - HTTP Request: POST https://azureai-prototyping-ai.openai.azure.com/openai/deployments/gpt-5-mini/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"


...

 ⚡  <<Intervention 7>> Triggered, effects are being applied... 
          > The trajectory explicitly provides N = 43 (the context states: "The last agent
          > simulation trajectory event number was 42, thus the current number of the NEXT POTENTIAL
          > TRAJECTORY EVENT is 43"). I then located the last TALK event where Anya introduced a
          > new, complete product/service: at event #34 she says: "Idea name: 'Marginalia
          > Exchange'..." (this is a full, standalone service description). Prior new-idea TALK
          > events by Anya were at #3/#14 ("The Conservator's Drawer") and #23 ("Errata Press").
          > After event #34 Anya's subsequent events are DONE (#35) and later comments/responses
          > (#39–#41) that critique or approve others' ideas or discuss implementation details (for
          > example at #39–#41 she evaluates Barbara's idea, drafts templates, and marks DONE), but
          > she does not propose another entirely new product/service idea after #34. The
          > proposition excludes refinements and variations; the later THINK/TALK entries are either
          > reflections or responses, not new idea proposals. Using M = 34 and N = 43 gives D = 9,
          > which is greater than 5. Therefore the statement "the agent has not proposed any new
          > product/service idea in the last 5 of his/her simulation trajectory events" is true for
          > Anya Calder-Mori: her last entirely new idea was at event 34, which is 9 events prior to
          > the current next event (43).  (confidence = 1.0)  Functional precondition was met.

2026-05-03 05:00:11,734 - MainThread(41284) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 05:00:11,760 - MainThread(41284) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-03 05:00:49,379 - MainThread(41284) - httpx - INFO - HTTP Request: POST https://azureai-prototyping-ai.openai.azure.com/openai/deployments/gpt-5-mini/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"


 ⚡  <<Intervention 8>> Triggered, effects are being applied... 
          > Detailed, concrete justification with trajectory evidence: - The proposition requires
          > identifying the last event where the agent proposed an "entirely new product/service
          > idea." The trajectory shows Barbara proposing ideas only at events where she issues a
          > TALK containing "Idea name: ...". - Specific events found:   - Event #22 (Date/time
          > identical to many events): Barbara Jean Pratt acts: [TALK] ->
          > "Idea name: 'Neighborhood Story Press'." This is a complete, self-contained idea (micro-
          > press, dropbox, chapbook workflow). It is a new product/service idea but not the last
          > one.   - Event #35: Barbara Jean Pratt acts: [TALK] -> "Idea
          > name: 'Community Memory Quilt'." This is another complete, distinct idea (volunteer-run
          > quilt program with numbered intake slips, booklets, laundering protocols). It is
          > explicitly marked as "Submitted a new, distinct idea" at event #36 (DONE follows #35
          > submission). The agent's own THOUGHT entries before #35 confirm she intended to propose
          > something entirely new and reviewed prior ideas (events #33-34). That shows the #35 idea
          > is indeed an intentional, entire new idea rather than a refinement. - After event #35
          > there are no Barbara TALK events that introduce a new idea. Events #36 is DONE for her
          > submission; #37–#45 include other agents' idea proposals and Barbara's commentary and
          > approvals (e.g., event #40 she THOUGHT about overlaps; #41 she TALKs a comment about
          > Anya's idea). None of those are new product/service proposals by Barbara. - The context
          > explicitly states the next potential event number is 46 (N = 46). Thus M = 35, N = 46,
          > so D = N - M = 11. - The proposition requires D > 5. Since 11 > 5, the proposition is
          > satisfied. - Also, the rule disallows counting "additional features, variations of or
          > other refinements to product/service ideas already proposed" as new. The two Barbara
          > ideas (#22 and #35) are distinct from each other and distinct from others', and #35 is
          > not a mere variation of her earlier #22 idea; it is a different concept (micro-press and
          > story dropbox vs. community textile quilt program). No later event shows a brand-new
          > idea from her. Therefore the last entirely new idea remains at event #35. - All of these
          > specific event numbers and contents are taken directly from the provided trajectory
          > (events #22 and #35 as cited, and the statement that last event number is 45, next is
          > 46). This yields D = 11 > 5, so the proposition is True. (confidence = 1.0)  Functional
          > precondition was met.

2026-05-03 05:00:51,771 - ThreadPoolExecutor-77_0(31364) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 05:00:51,795 - ThreadPoolExecutor-77_3(40256) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 05:00:51,831 - ThreadPoolExecutor-77_2(34460) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 05:00:51,860 - ThreadPoolExecutor-77_1(21988) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 05:00:51,889 - ThreadPoolExecutor-77_0(31364) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-03 05:00:51,897 - ThreadPoolExecutor-77_3(40256) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-03 05:00:51,925 - ThreadPoolExecutor-77_2(34460) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-03 05:00:51,951 - ThreadPoolExecutor-77_1(21988) - tinytroup

({'Hard Persona Adherence': [2,
   2,
   0,
   4,
   1,
   4,
   0,
   1,
   0,
   3,
   2,
   1,
   0,
   1,
   2,
   2,
   0,
   2,
   0,
   1,
   2,
   2,
   3,
   2,
   0,
   0,
   0,
   0,
   4,
   3,
   2,
   0,
   3,
   2,
   2,
   2,
   3,
   4,
   0,
   0],
  'Self-consistency': [9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   8,
   9,
   9,
   9,
   6,
   4,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   8,
   1,
   9,
   7,
   9,
   9,
   9],
  'Fluency': [8,
   7,
   8,
   8,
   7,
   8,
   7,
   8,
   6,
   6,
   6,
   5,
   7,
   7,
   1,
   7,
   6,
   6,
   7,
   8,
   6,
   6,
   2,
   3,
   3,
   1,
   8,
   7,
   8,
   2,
   7,
   7,
   7,
   6,
   2,
   1,
   4,
   6,
   7,
   7]},
 {'ideas_qty': [12, 12, 15, 12, 12, 14, 12, 12, 12],
  'Task Completion': [9, 9, 9, 9, 9, 8, 9, 9, 9, 9],
  'Divergence': [8, 2, 8, 4, 7, 6, 7, 9, 6, 4]})

In [18]:
brainstorm(people_groups[1], proposals_groups[0]) if len(people_groups) > 1  and len(proposals_groups) > 0 else None


############## STARTING A NEW RESEARCH SESSION #################
Overall experiment number: 1 / 4
Discussion objective: Create ideas for products or services that simplify, enhance, or bring joy to everyday tasks, routines, and interactions.
Trial number: 1
Agents: [TinyPerson(name='Colin Arthur Matthews'), TinyPerson(name='Colin Murray'), TinyPerson(name='Connor Walsh'), TinyPerson(name='Darren McCall')]
2026-05-03 05:06:26,047 - MainThread(41284) - tinytroupe - INFO - [TinyWorld 11] Running world simulation step 1 of 1.


──────────────────────────────────────────── TinyWorld 11 step 1 of 1 ─────────────────────────────────────────────

2026-05-03 05:06:26,068 - MainThread(41284) - tinytroupe - INFO - [TinyWorld 11] No timedelta provided, so the datetime was not advanced.
2026-05-03 05:06:27,077 - MainThread(41284) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 05:06:27,083 - MainThread(41284) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-03 05:06:57,365 - MainThread(41284) - httpx - INFO - HTTP Request: POST https://azureai-prototyping-ai.openai.azure.com/openai/deployments/gpt-5-mini/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-05-03 05:06:58,339 - MainThread(41284) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 05:06:58,346 - MainThread(41284) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-03 05:07:25,784 - MainThread(41284) - httpx - INFO - HTTP Request: POST https://azureai-prototyping-ai.openai.azure.com/openai/deployments/gpt

──────────────────────────────────────────── TinyWorld 11 step 1 of 5 ─────────────────────────────────────────────

2026-05-03 05:08:56,701 - MainThread(41284) - tinytroupe - INFO - [TinyWorld 11] No timedelta provided, so the datetime was not advanced.
2026-05-03 05:08:57,610 - MainThread(41284) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 05:08:57,627 - MainThread(41284) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-03 05:09:30,554 - MainThread(41284) - httpx - INFO - HTTP Request: POST https://azureai-prototyping-ai.openai.azure.com/openai/deployments/gpt-5-mini/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-05-03 05:09:31,493 - MainThread(41284) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 05:09:31,504 - MainThread(41284) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-03 05:10:03,050 - MainThread(41284) - httpx - INFO - HTTP Request: POST https://azureai-prototyping-ai.openai.azure.com/openai/deployments/gpt

...

 ⚡  <<Intervention 11>> Triggered, effects are being applied... 
          > The trajectory shows Connor Walsh proposed multiple distinct ideas; the latest distinct
          > new idea proposal is at event #34 where he 'acts: [TALK]' and states "Idea name:
          > 'PayPatch' — Shift‑Verified Micro‑Advance & Bill Buffer." That is a full, self-contained
          > product/service proposal (described, named, and followed by a DONE at #35). After event
          > #34 there are no further events in the trajectory where Connor puts forward another
          > entirely new product/service idea: events after #34 include #35 (DONE for the PayPatch
          > proposal), #36–#41 (other agents' idea submissions and Connor's THINK/TALK commentary on
          > others' ideas), and #42 (Darren's comment). The context explicitly gives the next
          > potential event number N = 43. Using M = 34, D = 43 - 34 = 9. Because the proposition
          > requires D to be strictly greater than 5 to be true, and 9 > 5, the proposition is true.
          > Concretely: Connor's last entirely new product/service idea proposal was at event 34
          > ('PayPatch'); nine events have elapsed since then (up to the next event number 43),
          > which is more than 5, so he has not proposed a completely new product/service idea in
          > the last 5 of his trajectory events. No counterexamples are present in the trajectory
          > (no later TALK events by Connor that introduce new named ideas), so the evaluation is
          > robust.  (confidence = 1.0)  Functional precondition was met.

2026-05-03 06:27:38,455 - MainThread(41284) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 06:27:38,544 - MainThread(41284) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-03 06:28:22,391 - MainThread(41284) - httpx - INFO - HTTP Request: POST https://azureai-prototyping-ai.openai.azure.com/openai/deployments/gpt-5-mini/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"


 ⚡  <<Intervention 12>> Triggered, effects are being applied... 
          > The proposition asks whether Darren has not proposed any entirely new product/service
          > idea in the last 5 of his simulation trajectory events, computed as D = N - M and true
          > iff D > 5. - N is given explicitly in the context as 46 (next potential event number).
          > - The last event where Darren actually proposed a new, self-contained product/service is
          > event #35: "Idea name: 'GrumbleBox'... A locked drop-box and postcard system..." This is
          > a complete, standalone idea, so it counts as an "entirely new product/service idea."  -
          > Darren previously proposed another distinct idea at event #22: "SpotFix," but that is
          > earlier than #35 and not the last.  - After #35 there are no Darren TALK events that
          > introduce a new idea. Darren's later activities include DONE (#36), other agents' idea
          > posts and comments, Darren's THINKs about others' ideas, and Darren's reaction to
          > PayPatch at #41 (a critique/endorsement/feedback), but no new idea proposals.  -
          > Therefore M = 35, and D = 46 - 35 = 11. Since 11 > 5, the condition in the proposition
          > is satisfied. Conclusion: The proposition is True because the agent's most recent
          > entirely new product/service idea was at event #35, and the current next event number is
          > 46, yielding D = 11 which is greater than 5. (confidence = 1.0)  Functional precondition
          > was met.

2026-05-03 06:28:24,423 - ThreadPoolExecutor-109_0(16996) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 06:28:24,443 - ThreadPoolExecutor-109_1(43152) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 06:28:24,465 - ThreadPoolExecutor-109_2(35388) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 06:28:24,472 - ThreadPoolExecutor-109_3(29496) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 06:28:24,528 - ThreadPoolExecutor-109_0(16996) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-03 06:28:24,551 - ThreadPoolExecutor-109_1(43152) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-03 06:28:24,571 - ThreadPoolExecutor-109_2(35388) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-03 06:28:24,584 - ThreadPoolExecutor-109_3(29496) - t

({'Hard Persona Adherence': [2,
   2,
   0,
   4,
   1,
   4,
   0,
   1,
   0,
   3,
   2,
   1,
   0,
   1,
   2,
   2,
   0,
   2,
   0,
   1,
   2,
   2,
   3,
   2,
   0,
   0,
   0,
   0,
   4,
   3,
   2,
   0,
   3,
   2,
   2,
   2,
   3,
   4,
   0,
   0,
   1,
   0,
   4,
   0,
   2,
   2,
   3,
   3,
   3,
   5,
   2,
   1,
   1,
   2,
   2,
   1],
  'Self-consistency': [9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   8,
   9,
   9,
   9,
   6,
   4,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   8,
   1,
   9,
   7,
   9,
   9,
   9,
   7,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9],
  'Fluency': [8,
   7,
   8,
   8,
   7,
   8,
   7,
   8,
   6,
   6,
   6,
   5,
   7,
   7,
   1,
   7,
   6,
   6,
   7,
   8,
   6,
   6,
   2,
   3,
   3,
   1,
   8,
   7,
   8,
   2,
   7,
   7,
   7,
   6,
   2,
   1,
   4,
   6,
   7,
   7,
   6,
   7,
   7,
   9,
   8,

In [19]:
brainstorm(people_groups[1], proposals_groups[1]) if len(people_groups) > 1  and len(proposals_groups) > 1 else None


############## STARTING A NEW RESEARCH SESSION #################
Overall experiment number: 1 / 6
Discussion objective: Explore ideas for products, services, or platforms that encourage curiosity, learning, adventure, and exploration of both the external world and inner self.
Trial number: 1
Agents: [TinyPerson(name='Colin Arthur Matthews'), TinyPerson(name='Colin Murray'), TinyPerson(name='Connor Walsh'), TinyPerson(name='Darren McCall')]
2026-05-03 06:34:38,339 - MainThread(41284) - tinytroupe - INFO - [TinyWorld 15] Running world simulation step 1 of 1.


──────────────────────────────────────────── TinyWorld 15 step 1 of 1 ─────────────────────────────────────────────

2026-05-03 06:34:38,348 - MainThread(41284) - tinytroupe - INFO - [TinyWorld 15] No timedelta provided, so the datetime was not advanced.
2026-05-03 06:34:39,403 - MainThread(41284) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 06:34:39,409 - MainThread(41284) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-03 06:35:09,174 - MainThread(41284) - httpx - INFO - HTTP Request: POST https://azureai-prototyping-ai.openai.azure.com/openai/deployments/gpt-5-mini/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-05-03 06:35:09,945 - MainThread(41284) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 06:35:09,950 - MainThread(41284) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-03 06:35:46,508 - MainThread(41284) - httpx - INFO - HTTP Request: POST https://azureai-prototyping-ai.openai.azure.com/openai/deployments/gpt

 ⚡  <<Intervention 14>> Triggered, effects are being applied... 
          > Concrete evidence from the trajectory that supports the decision: - The trajectory
          > explicitly lists only three events (numbers 0, 1, 2) and states the next event number is
          > 3 (N = 3). - Event #0: a USER -> Colin Murray conversation prompt (no agent output or
          > product/service idea by Colin). - Event #1: has Date and time: None and contains no
          > content showing Colin proposing a new product/service idea. - Event #2: again a USER ->
          > Colin Murray conversation prompt (no agent proposal). - There is no event in which Colin
          > Murray proposes any product or service idea (neither entirely new ideas nor
          > refinements). Because Colin never proposed any entirely new product/service idea in the
          > provided trajectory, he certainly has not proposed one within the last 5 of his
          > simulation trajectory events. Therefore the proposition (that he is not proposing new
          > product/service ideas anymore, per the criterion) is true for the given context.
          > (confidence = 1.0)  Functional precondition was met.

2026-05-03 06:35:47,301 - MainThread(41284) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 06:35:47,306 - MainThread(41284) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-03 06:36:16,830 - MainThread(41284) - httpx - INFO - HTTP Request: POST https://azureai-prototyping-ai.openai.azure.com/openai/deployments/gpt-5-mini/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-05-03 06:36:17,541 - MainThread(41284) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 06:36:17,547 - MainThread(41284) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-03 06:36:50,874 - MainThread(41284) - httpx - INFO - HTTP Request: POST https://azureai-prototyping-ai.openai.azure.com/openai/deployments/gpt-5-mini/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-05-03 06:36:52,972 - ThreadPoolExecutor-112_3(44120) - tiny

...

 ⚡  <<Intervention 15>> Triggered, effects are being applied... 
          > True, because the current next event number is N = 42 (context states last event was
          > #41). The last event where Connor Walsh proposed an entirely new, self-contained
          > product/service idea is event #33, where he proposed "ClaimCraft — Small-Claims Starter
          > Kit." No subsequent Connor events propose a new complete idea (later events are DONE,
          > THINK, or commentary about others' ideas, e.g., his procedural notes on Grumblebox at
          > #39 are refinements/comments, not a new product). Computing D = N - M = 42 - 33 = 9, and
          > since 9 > 5, the proposition's condition is satisfied. Therefore the statement that the
          > agent has not proposed any new product/service idea in the last 5 of his simulation
          > trajectory events is True. (confidence = 1.0)  Functional precondition was met.

2026-05-03 08:40:46,522 - MainThread(41284) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 08:40:46,565 - MainThread(41284) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-03 08:41:14,977 - MainThread(41284) - httpx - INFO - HTTP Request: POST https://azureai-prototyping-ai.openai.azure.com/openai/deployments/gpt-5-mini/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"


 ⚡  <<Intervention 16>> Triggered, effects are being applied... 
          > The proposition requires finding the last entirely new product/service idea proposed by
          > Darren and checking if the gap from that event to the current next event (47) is greater
          > than 5.  Concrete evidence from the trajectory: - N (current next event number) is
          > explicitly stated as 47 at the end of the transcript. - Darren's explicit new idea
          > proposals (TALK actions with idea names) in the trajectory are at the following event
          > numbers:   * Event #3 — TALK: Idea name: "Grudge Zine" (full proposal text). This is an
          > original idea.   * Event #14 — TALK: Idea name: "Grudge Zine" repeated. This is a
          > repetition (not a new distinct idea), so it does not count as a later new idea.   *
          > Event #23 — TALK: Idea name: 'Shoebox Post'. This is a distinct new idea and counts as a
          > new proposal at event 23.   * Event #36 — TALK: Idea name: 'Grumblebox'. This is a
          > distinct new idea and is the latest such TALK by Darren in the trace. - After event #36,
          > Darren has DONE at #37, and later at #41/#42 he comments on others' idea (ClaimCraft)
          > and does not propose a new product/service. Those later events are responses or
          > endorsements and explicitly not new idea proposals.  Applying the proposition's
          > computation steps: - M = 36 (last event Darren proposed an entirely new idea:
          > 'Grumblebox'). - D = N - M = 47 - 36 = 11. - The proposition requires D > 5. Since 11 >
          > 5, the condition is satisfied.  Therefore the proposition is True. Specific elements
          > that support this conclusion and reduce ambiguity: - The trajectory explicitly labels
          > event types (TALK, THINK, DONE), and Darren's TALK events at #3, #23, and #36 are
          > clearly idea proposals with unique names and full descriptions, while later Darren
          > events are either DONE or commentary (not new idea proposals). - Repeated segments
          > (e.g., #3 vs #14) are duplicates of the same idea and per the proposition should not be
          > treated as new distinct ideas; I treated them accordingly.  No contradictory evidence
          > appears in the trajectory: there is no Darren TALK after #36 that introduces a new idea.
          > Thus the difference D = 11 is reliably computed and the proposition holds. (confidence =
          > 1.0)  Functional precondition was met.

2026-05-03 08:41:17,353 - ThreadPoolExecutor-157_0(38436) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 08:41:17,426 - ThreadPoolExecutor-157_0(38436) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-03 08:41:17,454 - ThreadPoolExecutor-157_1(48740) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 08:41:17,470 - ThreadPoolExecutor-157_3(31664) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 08:41:17,475 - ThreadPoolExecutor-157_2(15248) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 08:41:17,535 - ThreadPoolExecutor-157_1(48740) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-03 08:41:17,547 - ThreadPoolExecutor-157_3(31664) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-03 08:41:17,554 - ThreadPoolExecutor-157_2(15248) - t

({'Hard Persona Adherence': [2,
   2,
   0,
   4,
   1,
   4,
   0,
   1,
   0,
   3,
   2,
   1,
   0,
   1,
   2,
   2,
   0,
   2,
   0,
   1,
   2,
   2,
   3,
   2,
   0,
   0,
   0,
   0,
   4,
   3,
   2,
   0,
   3,
   2,
   2,
   2,
   3,
   4,
   0,
   0,
   1,
   0,
   4,
   0,
   2,
   2,
   3,
   3,
   3,
   5,
   2,
   1,
   1,
   2,
   2,
   1,
   4,
   0,
   1,
   4,
   0,
   2,
   2,
   3,
   2,
   0,
   1,
   3,
   0,
   3,
   0,
   3,
   6,
   2,
   1,
   0,
   1,
   2,
   2,
   3],
  'Self-consistency': [9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   8,
   9,
   9,
   9,
   6,
   4,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   8,
   1,
   9,
   7,
   9,
   9,
   9,
   7,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,

In [20]:
brainstorm(people_groups[2], proposals_groups[0]) if len(people_groups) > 2  and len(proposals_groups) > 0 else None


############## STARTING A NEW RESEARCH SESSION #################
Overall experiment number: 1 / 4
Discussion objective: Create ideas for products or services that simplify, enhance, or bring joy to everyday tasks, routines, and interactions.
Trial number: 1
Agents: [TinyPerson(name='Dean Bartlett'), TinyPerson(name='Declan Blackwell'), TinyPerson(name='Edgar Milton Crane'), TinyPerson(name='Leonard Victor Hale')]
2026-05-03 08:48:00,247 - MainThread(41284) - tinytroupe - INFO - [TinyWorld 21] Running world simulation step 1 of 1.


──────────────────────────────────────────── TinyWorld 21 step 1 of 1 ─────────────────────────────────────────────

2026-05-03 08:48:00,252 - MainThread(41284) - tinytroupe - INFO - [TinyWorld 21] No timedelta provided, so the datetime was not advanced.
2026-05-03 08:48:01,255 - MainThread(41284) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 08:48:01,264 - MainThread(41284) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-03 08:48:27,662 - MainThread(41284) - httpx - INFO - HTTP Request: POST https://azureai-prototyping-ai.openai.azure.com/openai/deployments/gpt-5-mini/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-05-03 08:48:28,373 - MainThread(41284) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 08:48:28,377 - MainThread(41284) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-03 08:48:53,005 - MainThread(41284) - httpx - INFO - HTTP Request: POST https://azureai-prototyping-ai.openai.azure.com/openai/deployments/gpt

──────────────────────────────────────────── TinyWorld 21 step 1 of 5 ─────────────────────────────────────────────

2026-05-03 08:50:12,946 - MainThread(41284) - tinytroupe - INFO - [TinyWorld 21] No timedelta provided, so the datetime was not advanced.
2026-05-03 08:50:14,356 - MainThread(41284) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 08:50:14,383 - MainThread(41284) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-03 08:50:45,093 - MainThread(41284) - httpx - INFO - HTTP Request: POST https://azureai-prototyping-ai.openai.azure.com/openai/deployments/gpt-5-mini/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-05-03 08:50:45,748 - MainThread(41284) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 08:50:45,761 - MainThread(41284) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-03 08:51:18,995 - MainThread(41284) - httpx - INFO - HTTP Request: POST https://azureai-prototyping-ai.openai.azure.com/openai/deployments/gpt

...

 ⚡  <<Intervention 19>> Triggered, effects are being applied... 
          > The current next event number N is 44 (context explicitly states this). The most recent
          > event in which Edgar himself proposed an entirely new product/service idea is event #34,
          > where Edgar 'TALK' introduces "Idea name: 'Supplier Reliability Registry'" — a
          > community-run, evidence-backed register and monthly Reliability Report. Earlier Edgar
          > proposed other ideas (e.g., event #21 "Complaint Concierge & Support Collective"), but
          > the last entirely new idea he proposed is at #34. After #34 Edgar's subsequent actions
          > are: marking DONE at #35, then later he THINKs at #39 about Leonard's Doorstep Docket
          > and at #40 he commits to hosting a pilot for Leonard's Doorstep Docket — however,
          > Doorstep Docket was proposed by Leonard at event #38, so Edgar's #40 message is an
          > implementation/hosting pledge, not a new, distinct product/service idea. There is no
          > later Edgar event that introduces a completely new product/service idea. Therefore M =
          > 34. With N = 44, D = 10, and since 10 > 5 the proposition's condition is satisfied. All
          > relevant trajectory events referenced: #21 (Edgar new idea earlier), #34 (Edgar's last
          > entirely new idea), #35 (Edgar DONE after #34), #38 (Leonard's Doorstep Docket), #39
          > (Edgar THINK), #40 (Edgar hosts pilot for Leonard's idea). These concrete event
          > references support the calculation and conclusion. (confidence = 1.0)  Functional
          > precondition was met.

2026-05-03 10:29:09,658 - MainThread(41284) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 10:29:09,697 - MainThread(41284) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-03 10:29:36,765 - MainThread(41284) - httpx - INFO - HTTP Request: POST https://azureai-prototyping-ai.openai.azure.com/openai/deployments/gpt-5-mini/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"


 ⚡  <<Intervention 20>> Triggered, effects are being applied... 
          > True, because:  - The context explicitly gives the next event number N = 46 (the
          > trajectory ended at event #45).  - The last time Leonard proposed a brand-new, self-
          > contained product/service was at event #35, where he posted "Idea name: 'Doorstep Docket
          > — Scheduled Porch Checks & Paper Triage'" (a full idea description). That is a clear,
          > standalone new idea.  - Earlier he also proposed a different idea at event #22 ('The
          > Ledger Club'), but #35 is later and therefore is the most recent new-idea event (M =
          > 35).  - Subsequent Leonard entries (events #36 onward) are 'DONE', comments, thoughts,
          > or approvals (for example, event #40 is a THINK about Edgar's registry and event #41 is
          > Leonard's TALK giving implementation notes). None of these are new, entirely distinct
          > product/service proposals — they are refinements, reactions, or operational notes, which
          > the proposition explicitly excludes.  - Compute D = 46 - 35 = 11, and 11 > 5 holds.
          > Therefore the statement that "the agent has not proposed any new product/service idea in
          > the last 5 of his/her simulation trajectory events" (i.e., the last new idea was more
          > than 5 events ago) is true. (confidence = 1.0)  Functional precondition was met.

2026-05-03 10:29:39,914 - ThreadPoolExecutor-189_3(4572) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 10:29:39,925 - ThreadPoolExecutor-189_1(25448) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 10:29:39,938 - ThreadPoolExecutor-189_2(49984) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 10:29:39,940 - ThreadPoolExecutor-189_0(2124) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 10:29:40,065 - ThreadPoolExecutor-189_3(4572) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-03 10:29:40,102 - ThreadPoolExecutor-189_1(25448) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-03 10:29:40,106 - ThreadPoolExecutor-189_2(49984) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-03 10:29:40,124 - ThreadPoolExecutor-189_0(2124) - tinyt

({'Hard Persona Adherence': [2,
   2,
   0,
   4,
   1,
   4,
   0,
   1,
   0,
   3,
   2,
   1,
   0,
   1,
   2,
   2,
   0,
   2,
   0,
   1,
   2,
   2,
   3,
   2,
   0,
   0,
   0,
   0,
   4,
   3,
   2,
   0,
   3,
   2,
   2,
   2,
   3,
   4,
   0,
   0,
   1,
   0,
   4,
   0,
   2,
   2,
   3,
   3,
   3,
   5,
   2,
   1,
   1,
   2,
   2,
   1,
   4,
   0,
   1,
   4,
   0,
   2,
   2,
   3,
   2,
   0,
   1,
   3,
   0,
   3,
   0,
   3,
   6,
   2,
   1,
   0,
   1,
   2,
   2,
   3,
   3,
   1,
   0,
   3,
   3,
   3,
   0,
   3,
   0,
   3,
   1,
   0,
   1,
   3,
   3,
   0],
  'Self-consistency': [9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   8,
   9,
   9,
   9,
   6,
   4,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   8,
   1,
   9,
   7,
   9,
   9,
   9,
   7,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,

In [21]:
brainstorm(people_groups[2], proposals_groups[1]) if len(people_groups) > 2  and len(proposals_groups) > 1 else None


############## STARTING A NEW RESEARCH SESSION #################
Overall experiment number: 1 / 6
Discussion objective: Explore ideas for products, services, or platforms that encourage curiosity, learning, adventure, and exploration of both the external world and inner self.
Trial number: 1
Agents: [TinyPerson(name='Dean Bartlett'), TinyPerson(name='Declan Blackwell'), TinyPerson(name='Edgar Milton Crane'), TinyPerson(name='Leonard Victor Hale')]
2026-05-03 10:36:01,512 - MainThread(41284) - tinytroupe - INFO - [TinyWorld 25] Running world simulation step 1 of 1.


──────────────────────────────────────────── TinyWorld 25 step 1 of 1 ─────────────────────────────────────────────

2026-05-03 10:36:01,525 - MainThread(41284) - tinytroupe - INFO - [TinyWorld 25] No timedelta provided, so the datetime was not advanced.
2026-05-03 10:36:03,358 - MainThread(41284) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 10:36:03,367 - MainThread(41284) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-03 10:36:36,325 - MainThread(41284) - httpx - INFO - HTTP Request: POST https://azureai-prototyping-ai.openai.azure.com/openai/deployments/gpt-5-mini/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-05-03 10:36:37,238 - MainThread(41284) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 10:36:37,246 - MainThread(41284) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-03 10:37:13,710 - MainThread(41284) - httpx - INFO - HTTP Request: POST https://azureai-prototyping-ai.openai.azure.com/openai/deployments/gpt

──────────────────────────────────────────── TinyWorld 25 step 1 of 5 ─────────────────────────────────────────────

2026-05-03 10:38:44,415 - MainThread(41284) - tinytroupe - INFO - [TinyWorld 25] No timedelta provided, so the datetime was not advanced.
2026-05-03 10:38:45,610 - MainThread(41284) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 10:38:45,632 - MainThread(41284) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-03 10:39:15,987 - MainThread(41284) - httpx - INFO - HTTP Request: POST https://azureai-prototyping-ai.openai.azure.com/openai/deployments/gpt-5-mini/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-05-03 10:39:17,376 - MainThread(41284) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 10:39:17,403 - MainThread(41284) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-03 10:40:00,767 - MainThread(41284) - httpx - INFO - HTTP Request: POST https://azureai-prototyping-ai.openai.azure.com/openai/deployments/gpt

...

 ⚡  <<Intervention 23>> Triggered, effects are being applied... 
          > The trajectory explicitly lists events and contents. The current next event number is
          > given as 42 (the log states last event 41 and next potential event 42). Edgar's explicit
          > new idea posts are at event #22 ('Idea name: "Fix‑It Storyboard"') and at event #33
          > ('Idea name: "ProofPost Drop"'). After #33 Edgar's later entries (#38 thinking, #39
          > TALK) are comments and suggested non-new-idea refinements (staff script, pricing,
          > templates) or DONE statuses; these are not new, self-contained product/service ideas per
          > the proposition's definition. Therefore the most recent entirely new idea was at event
          > 33, producing D = 42 - 33 = 9, which is greater than 5. That satisfies the proposition's
          > condition that the agent has not proposed any new product/service idea in the last 5 of
          > his simulation trajectory events. (confidence = 1.0)  Functional precondition was met.

2026-05-03 12:40:51,089 - MainThread(41284) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 12:40:51,119 - MainThread(41284) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-03 12:41:13,413 - MainThread(41284) - httpx - INFO - HTTP Request: POST https://azureai-prototyping-ai.openai.azure.com/openai/deployments/gpt-5-mini/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"


 ⚡  <<Intervention 24>> Triggered, effects are being applied... 
          > The proposition states Leonard has not proposed any entirely new product/service idea in
          > the last 5 of his/her simulation trajectory events, i.e., D = N - M must be greater than
          > 5. From the trajectory: N = 45 (next event number). Leonard’s explicit new idea posts
          > occurred at event #21 (Neighborhood Press — a paper kit) and event #34 (Stamped Dispatch
          > — a citizen mailroom/delivery service). After event #34, Leonard’s subsequent actions
          > are marked as DONE, THINK, or replies/feedback (events #35–#44) and do not include any
          > new “Idea name:” proposals. Therefore M = 34 and D = 45 - 34 = 11. Because 11 is greater
          > than 5, the proposition is true. Concretely: the last entirely new idea was proposed at
          > event 34 (Stamped Dispatch), and 11 events have occurred (or are numbered) since then
          > (up to the next event number 45), which exceeds the required gap of 5. No later event in
          > the trajectory shows Leonard introducing a completely new product/service idea (only
          > comments, critiques, and implementation tweaks), so the condition holds. (confidence =
          > 1.0)  Functional precondition was met.

2026-05-03 12:41:15,911 - ThreadPoolExecutor-237_3(10236) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 12:41:15,978 - ThreadPoolExecutor-237_2(22256) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 12:41:16,008 - ThreadPoolExecutor-237_3(10236) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-03 12:41:16,080 - ThreadPoolExecutor-237_2(22256) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-03 12:41:16,089 - ThreadPoolExecutor-237_1(17972) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 12:41:16,114 - ThreadPoolExecutor-237_0(47940) - tinytroupe - INFO - Using Azure OpenAI Service API with key...
2026-05-03 12:41:16,186 - ThreadPoolExecutor-237_1(17972) - tinytroupe - INFO - Waiting 5.0 seconds before next API request (to avoid throttling)...
2026-05-03 12:41:16,216 - ThreadPoolExecutor-237_0(47940) - t

({'Hard Persona Adherence': [2,
   2,
   0,
   4,
   1,
   4,
   0,
   1,
   0,
   3,
   2,
   1,
   0,
   1,
   2,
   2,
   0,
   2,
   0,
   1,
   2,
   2,
   3,
   2,
   0,
   0,
   0,
   0,
   4,
   3,
   2,
   0,
   3,
   2,
   2,
   2,
   3,
   4,
   0,
   0,
   1,
   0,
   4,
   0,
   2,
   2,
   3,
   3,
   3,
   5,
   2,
   1,
   1,
   2,
   2,
   1,
   4,
   0,
   1,
   4,
   0,
   2,
   2,
   3,
   2,
   0,
   1,
   3,
   0,
   3,
   0,
   3,
   6,
   2,
   1,
   0,
   1,
   2,
   2,
   3,
   3,
   1,
   0,
   3,
   3,
   3,
   0,
   3,
   0,
   3,
   1,
   0,
   1,
   3,
   3,
   0,
   1,
   1,
   1,
   6,
   0,
   0,
   4,
   1,
   1,
   3,
   2,
   5,
   3,
   2,
   0,
   0,
   1,
   3,
   3,
   3,
   2,
   3,
   0,
   1],
  'Self-consistency': [9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   8,
   9,
   9,
   9,
   6,
   4,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   9,
   8,
   1,
   9,
   7,
   9,
   9,

In [22]:
brainstorm(people_groups[3], proposals_groups[0]) if len(people_groups) > 3  and len(proposals_groups) > 0 else None

In [23]:
brainstorm(people_groups[3], proposals_groups[1]) if len(people_groups) > 3  and len(proposals_groups) > 1 else None

In [24]:
brainstorm(people_groups[4], proposals_groups[0]) if len(people_groups) > 4  and len(proposals_groups) > 0 else None

In [25]:
brainstorm(people_groups[4], proposals_groups[1]) if len(people_groups) > 4  and len(proposals_groups) > 1 else None

## Extract results and analyze

In [26]:
if experiment_runner.get_active_experiment() in ["Control", "Treatment"]:
    combined_scores = {**agent_propositions_scores, **environment_propositions_scores}
    experiment_runner.add_experiment_results(combined_scores, experiment_name=experiment_runner.get_active_experiment()) 
    
    plot_scores(combined_scores)

else:
    print("Experiment finished. No more experiments to run.")

{'Divergence': [8,
                2,
                8,
                4,
                7,
                6,
                7,
                9,
                6,
                4,
                7,
                8,
                9,
                6,
                6,
                7,
                7,
                8,
                6,
                8,
                2,
                8,
                6,
                9,
                4,
                8,
                9,
                6,
                7,
                2],
 'Fluency': [8,
             7,
             8,
             8,
             7,
             8,
             7,
             8,
             6,
             6,
             6,
             5,
             7,
             7,
             1,
             7,
             6,
             6,
             7,
             8,
             6,
             6,
             2,
             3,
             3,
             1,
             

,Proposition,Average Score,Standard Deviation,Count
0,Hard Persona Adherence,1.766667,1.447881,120.0
1,Self-consistency,8.650000,1.394166,120.0
2,Fluency,6.183333,1.777466,120.0
3,ideas_qty,12.551724,0.827484,29.0
4,Task Completion,8.966667,0.182574,30.0
5,Divergence,6.466667,2.063364,30.0


In [27]:
if experiment_runner.has_finished_all_experiments():
    print("All experiments have been finished.")
    print(f"STATISTICTS: Control vs")
    pprint(experiment_runner.run_statistical_tests(control_experiment_name='Control'))

    # plot scores of both experiments
    experiment_control_scores = experiment_runner.get_experiment_results("Control")
    experiment_treatment_scores = experiment_runner.get_experiment_results("Treatment")
    
    
    plot_scores(experiment_control_scores)
    plot_scores(experiment_treatment_scores)

else:
    print("Not all experiments have been finished. RESTART AND RERUN.")

Not all experiments have been finished. RESTART AND RERUN.


In [28]:
experiment_runner.finish_active_experiment()

2026-05-03 12:47:35,848 - MainThread(41284) - tinytroupe - INFO - Experiment 'Treatment' marked as finished.
2026-05-03 12:47:35,852 - MainThread(41284) - tinytroupe - INFO - All experiments have been finished.


True